# Gemma 4 31B Q3 PoC - 配当変更(増減配) テスト

Colab無料枠 T4 (16GB) + Ollama + Gemma 31B Q3量子化

**ランタイム**: T4 GPU（ランタイム → ランタイムのタイプを変更 → T4 GPU）

In [1]:
# Step 1: GPU確認
!nvidia-smi

Mon Apr 13 22:55:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Step 2: zstd + Ollama インストール + 起動
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, os
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
proc = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print('Ollama started')

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (481 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently i

In [3]:
# Step 3: Gemma 31B Q3 プル（~14GB）
# T4 16GBにギリギリ収まる
!ollama pull gemma4:31b-it-q3_K_M
MODEL_NAME = 'gemma4:31b-it-q3_K_M'


Error: pull model manifest: file does not exist


In [4]:
# Step 3-FB: T4に収まらない場合のフォールバック（Q2_K ~12GB）
# !ollama pull gemma4:31b-it-q2_K
# MODEL_NAME = 'gemma4:31b-it-q2_K'

In [5]:
# Step 4: 動作確認
import requests, json

def call_ollama(prompt, model=None):
    model = model or MODEL_NAME
    resp = requests.post(
        'http://localhost:11434/api/generate',
        json={'model': model, 'prompt': prompt, 'stream': False,
              'options': {'temperature': 0.1, 'num_predict': 500}},
        timeout=300,
    )
    return resp.json()['response']

print(call_ollama('Say hello in Japanese. 5 words.'))

KeyError: 'response'

In [ ]:
# Step 5: Google Drive マウント + GCPキー取得
from google.colab import drive
drive.mount('/content/drive')

import os
SA_KEY = None
for p in [
    '/content/drive/MyDrive/claude/investment-agent/keys/gcp-service-account.json',
    '/content/drive/MyDrive/keys/gcp-service-account.json',
    '/content/drive/MyDrive/gcp-service-account.json',
]:
    if os.path.exists(p):
        SA_KEY = p
        print(f'Found: {SA_KEY}')
        break

if not SA_KEY:
    print('Upload gcp-service-account.json manually:')
    from google.colab import files
    uploaded = files.upload()
    SA_KEY = list(uploaded.keys())[0]

In [ ]:
# Step 6: BigQueryクライアント
!pip install google-cloud-bigquery -q

from google.oauth2 import service_account
from google.cloud import bigquery
import re, time

creds = service_account.Credentials.from_service_account_file(
    SA_KEY, scopes=['https://www.googleapis.com/auth/cloud-platform'])
bq = bigquery.Client(project='gmailpj-357912', credentials=creds)
print('BQ client ready')

In [ ]:
# Step 7: 配当変更テスト対象データ取得
sql = '''
WITH doc_texts AS (
    SELECT
        DOC_ID, TICKER, DOC_TITLE, MAIN_CATEGORY, SUB_CATEGORIES, TEXT_LENGTH,
        STRING_AGG(CHUNK_TEXT, '' ORDER BY CHUNK_TEXT) AS full_text
    FROM `gmailpj-357912.STOCK.TDNET_DOCUMENTS_ENHANCED`
    WHERE SUBMISSION_DATE BETWEEN '2024-01-01' AND '2024-01-31'
      AND MAIN_CATEGORY IN ('決算短信', '決算説明資料')
    GROUP BY DOC_ID, TICKER, DOC_TITLE, MAIN_CATEGORY, SUB_CATEGORIES, TEXT_LENGTH
)
SELECT * FROM doc_texts
WHERE '配当変更（増減配）' IN UNNEST(SUB_CATEGORIES)
   OR '配当' IN UNNEST(SUB_CATEGORIES)
ORDER BY TICKER, DOC_ID
'''
rows = list(bq.query(sql).result())
print(f'Loaded {len(rows)} docs')
haito_henkou_count = sum(1 for r in rows if '配当変更（増減配）' in (r.SUB_CATEGORIES or []))
print(f'  うち配当変更あり: {haito_henkou_count}件')

In [ ]:
# Step 8: プロンプト（v5と同一、配当変更の定型表現指示あり）
VALID_CATEGORIES = [
    '決算短信', '決算説明資料', '業績修正', '業績予想', '月次開示',
    '配当', '配当変更（増減配）', '株主優待', '自己株式取得', '自己株式消却',
    '株式分割・併合', '第三者割当・公募増資', '新株予約権発行', '株式売出し',
    'TOB・MBO', '子会社化・買収', '資産売却（不動産）',
    '合併・組織再編', '会社分割', '上場廃止', '主要株主異動',
    '大型受注・契約', '提携・協業', '事業計画（グロース）', '中期経営計画',
    'リストラ・希望退職', '特別利益', '特別損失',
    '役員異動（代表クラス）', '監査人異動',
    'インシデント（セキュリティ）', '訴訟・法的手続き',
    '受注高/受注残高', '業績の重要な先行指標',
]
VALID_CATEGORIES_STR = '\n'.join(f'- {c}' for c in VALID_CATEGORIES)

def build_prompt(doc_title, text):
    return f'''以下のTDnet適時開示文書から、投資判断に関連する情報カテゴリをすべて抽出してください。

★重要ルール:
- 具体的な事実・実績・数値・施策として記載されている情報カテゴリを抽出すること
- 配当金額・合併・提携など、具体的な施策や金額が記載されていれば必ず含めること
- 以下のカテゴリ名から選択すること（一言一句違わず出力）:
{VALID_CATEGORIES_STR}

★カテゴリ判定の具体的基準:
- 「配当変更（増減配）」: 以下のいずれかに該当すれば必ず選択:
  - 決算短信の「直近に公表されている配当予想からの修正の有無：有」の記載
  - 「増配」「減配」「配当予想の修正」「記念配当」「特別配当」の文言
  - 前期比・前回予想比で配当金額が変更されている記載
  「配当」とは別に独立して選択すること

文書タイトル: {doc_title}
テキスト: {text[:20000]}

【出力形式】必ず単一のJSONオブジェクトのみを返してください。
{{ "sub_categories": ["カテゴリ1", "カテゴリ2"] }}'''

print('Prompt ready')

In [ ]:
# Step 9: テスト実行
results = []
start = time.time()
for i, row in enumerate(rows):
    prompt = build_prompt(row.DOC_TITLE, row.full_text)
    try:
        raw = call_ollama(prompt)
        cleaned = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
        m = re.search(r'\{[^{}]*"sub_categories"[^{}]*\}', cleaned, flags=re.DOTALL)
        if m:
            parsed = json.loads(m.group())
        else:
            parsed = json.loads(cleaned)
        subs = parsed.get('sub_categories', [])
    except Exception as e:
        subs = []
        print(f'  ERR {row.TICKER}: {str(e)[:80]}')

    gemini_subs = list(row.SUB_CATEGORIES) if row.SUB_CATEGORIES else []
    results.append({
        'ticker': row.TICKER,
        'doc_title': row.DOC_TITLE[:40],
        'gemini_subs': gemini_subs,
        'gemma31b_subs': subs,
        'gemini_haito_henkou': '配当変更（増減配）' in gemini_subs,
        'gemma31b_haito_henkou': '配当変更（増減配）' in subs,
    })

    if (i+1) % 10 == 0:
        elapsed = time.time() - start
        eta = elapsed / (i+1) * (len(rows) - i - 1)
        print(f'{i+1}/{len(rows)} ({elapsed:.0f}s 経過, 残り{eta:.0f}s)')

total_time = time.time() - start
print(f'\nDone: {len(results)} docs in {total_time:.0f}s')

In [ ]:
# Step 10: 結果集計
from collections import Counter

patterns = Counter()
for r in results:
    patterns[(r['gemini_haito_henkou'], r['gemma31b_haito_henkou'])] += 1

print('=== 配当変更（増減配） Gemini vs Gemma31B Q3 ===')
print(f'{"Gemini":<10} {"31B Q3":<10} {"件数"}')
for (g, q), cnt in sorted(patterns.items(), key=lambda x: -x[1]):
    print(f'{"○" if g else "×":<10} {"○" if q else "×":<10} {cnt}')

g_yes = sum(1 for r in results if r['gemini_haito_henkou'])
q_yes = sum(1 for r in results if r['gemma31b_haito_henkou'])
both_yes = sum(1 for r in results if r['gemini_haito_henkou'] and r['gemma31b_haito_henkou'])
match = sum(1 for r in results if r['gemini_haito_henkou'] == r['gemma31b_haito_henkou'])

print(f'\nGemini検出: {g_yes}/{len(results)}')
print(f'31B Q3検出: {q_yes}/{len(results)}')
print(f'一致: {match}/{len(results)} ({match/len(results)*100:.1f}%)')

with open('/content/drive/MyDrive/gemma31b_q3_haito.json', 'w') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print('\nSaved: /content/drive/MyDrive/gemma31b_q3_haito.json')

In [ ]:
# Step 11: LINE通知
NTFY_TOPIC = 'inv_agent_5c2003835f9f49e0'
msg = f'''Colab Gemma31B Q3 完了
{len(results)}件テスト
配当変更検出:
  Gemini: {g_yes}件
  31B Q3: {q_yes}件
一致率: {match}/{len(results)} ({match/len(results)*100:.1f}%)
処理時間: {total_time:.0f}s'''

resp = requests.post(
    f'https://ntfy.sh/{NTFY_TOPIC}',
    data=msg.encode('utf-8'),
    headers={'Title': 'Colab Gemma31B Q3完了'},
)
print(f'LINE: {resp.status_code}')